In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from scipy.signal import welch
import os
import glob

In [5]:
class RGBMentalPerformancePredictor:
    """
    A model to predict mental performance (Au) from RGB signal time series data.
    """

    def __init__(self, r_path, g_path, b_path):
        self.r_path = r_path
        self.g_path = g_path
        self.b_path = b_path
        self.channel_paths = {'R': r_path, 'G': g_path, 'B': b_path}
        self.scaler = StandardScaler()
        self.model = None

    def load_data(self):
        """Load and concatenate RGB time series chunk data from separate R, G, B folders."""
        print("Loading data from RGB channels...")

        data_list = []

        # Get all unique participant identifiers from chunk filenames
        # Check R channel for participant IDs
        chunk_files = glob.glob(os.path.join(self.r_path, "*chunks*.csv"))

        # Extract participant IDs from filenames
        participant_ids = set()
        for file in chunk_files:
            filename = os.path.basename(file)
            # Extract number from filename like "R_chunks_3.csv" or "R_chunks_10.csv"
            import re
            match = re.search(r'chunks[_]?(\d+)', filename)
            if match:
                participant_ids.add(match.group(1))

        participant_ids = sorted(list(participant_ids))
        print(f"Found {len(participant_ids)} participants: {participant_ids}")

        # Process each participant
        for participant_id in participant_ids:
            print(f"Processing participant {participant_id}...")

            # Load and concatenate chunks from all three channels
            channel_data = {}

            for channel_name, channel_path in self.channel_paths.items():
                # Find all chunk files for this participant in this channel
                pattern = os.path.join(channel_path, f"*chunks*{participant_id}.csv")
                chunk_files = sorted(glob.glob(pattern))

                if not chunk_files:
                    print(f"  Warning: No chunks found for participant {participant_id} in channel {channel_name}")
                    continue

                print(f"  Channel {channel_name}: {len(chunk_files)} chunk(s)")

                # Load and concatenate all chunks for this participant
                chunk_dfs = []
                for chunk_file in chunk_files:
                    df_chunk = pd.read_csv(chunk_file)
                    chunk_dfs.append(df_chunk)

                # Concatenate all chunks
                channel_data[channel_name] = pd.concat(chunk_dfs, ignore_index=True)
                print(f"    Total rows: {len(channel_data[channel_name])}")

            # Verify all channels have same number of rows
            if len(channel_data) == 3:
                lengths = [len(channel_data[ch]) for ch in ['R', 'G', 'B']]
                if len(set(lengths)) != 1:
                    print(f"  Warning: Channels have different lengths: {lengths}")
                    min_length = min(lengths)
                    print(f"  Trimming to minimum length: {min_length}")
                    for ch in ['R', 'G', 'B']:
                        channel_data[ch] = channel_data[ch].iloc[:min_length]

            # Extract features for each row
            num_rows = len(channel_data['R'])
            for idx in range(num_rows):
                features = {}

                # Get time series data (columns t0-t59) and target (AU)
                for channel_name in ['R', 'G', 'B']:
                    # Get time series columns (t0, t1, ..., t59)
                    time_series_cols = [f't{i}' for i in range(60)]
                    time_series = channel_data[channel_name].iloc[idx][time_series_cols].values.astype(float)

                    # Extract features from this channel
                    channel_features = self.extract_features(time_series, channel_name)
                    features.update(channel_features)

                # Get target value (AU) from any channel (should be same across channels)
                try:
                    target = channel_data['R'].iloc[idx]['AU']
                    # Skip if target is NaN
                    if pd.isna(target):
                        continue
                except (KeyError, IndexError):
                    print(f"  Warning: Could not get AU value for row {idx}")
                    continue

                features['participant_id'] = participant_id
                features['target'] = target
                data_list.append(features)

        self.df = pd.DataFrame(data_list)
        print(f"\n✓ Successfully loaded {len(self.df)} total samples from {len(participant_ids)} participants")
        return self.df

    def extract_features(self, time_series, channel_name):
        """Extract statistical, temporal, and frequency features from time series."""
        features = {}
        prefix = f"{channel_name}_"

        # Statistical features
        features[f'{prefix}mean'] = np.mean(time_series)
        features[f'{prefix}std'] = np.std(time_series)
        features[f'{prefix}min'] = np.min(time_series)
        features[f'{prefix}max'] = np.max(time_series)
        features[f'{prefix}median'] = np.median(time_series)
        features[f'{prefix}range'] = np.ptp(time_series)
        features[f'{prefix}skewness'] = stats.skew(time_series)
        features[f'{prefix}kurtosis'] = stats.kurtosis(time_series)

        # Percentiles
        features[f'{prefix}p25'] = np.percentile(time_series, 25)
        features[f'{prefix}p75'] = np.percentile(time_series, 75)
        features[f'{prefix}iqr'] = features[f'{prefix}p75'] - features[f'{prefix}p25']

        # Temporal features
        diff = np.diff(time_series)
        features[f'{prefix}mean_diff'] = np.mean(diff)
        features[f'{prefix}std_diff'] = np.std(diff)
        features[f'{prefix}mean_abs_diff'] = np.mean(np.abs(diff))

        # Zero crossing rate
        features[f'{prefix}zero_crossing'] = np.sum(np.diff(np.sign(time_series - np.mean(time_series))) != 0)

        # Trend (linear regression slope)
        x = np.arange(len(time_series))
        slope, intercept, _, _, _ = stats.linregress(x, time_series)
        features[f'{prefix}trend_slope'] = slope
        features[f'{prefix}trend_intercept'] = intercept

        # Frequency domain features (Power Spectral Density)
        freqs, psd = welch(time_series, fs=1.0, nperseg=min(len(time_series), 30))
        features[f'{prefix}psd_mean'] = np.mean(psd)
        features[f'{prefix}psd_std'] = np.std(psd)
        features[f'{prefix}psd_max'] = np.max(psd)
        features[f'{prefix}dominant_freq'] = freqs[np.argmax(psd)]

        # Energy
        features[f'{prefix}energy'] = np.sum(time_series ** 2)

        return features

    def prepare_data(self):
        """Prepare features and target for modeling."""
        print("\nPreparing data...")

        # Check for missing values
        print(f"Total samples before cleaning: {len(self.df)}")
        print(f"Missing values in target: {self.df['target'].isna().sum()}")

        # Remove rows with missing target values
        self.df = self.df.dropna(subset=['target'])
        print(f"Total samples after removing missing targets: {len(self.df)}")

        # Exclude non-feature columns
        feature_cols = [col for col in self.df.columns
                       if col not in ['target', 'participant_id']]

        X = self.df[feature_cols]
        y = self.df['target']

        # Check for missing values in features
        missing_features = X.isna().sum()
        if missing_features.sum() > 0:
            print(f"\nWarning: Found {missing_features.sum()} missing values in features")
            print("Filling missing values with column means...")
            X = X.fillna(X.mean())

        # Check for infinite values
        inf_mask = np.isinf(X.values)
        if inf_mask.any():
            print(f"Warning: Found {inf_mask.sum()} infinite values in features")
            print("Replacing infinite values with column max/min...")
            X = X.replace([np.inf, -np.inf], np.nan)
            X = X.fillna(X.mean())

        print(f"\n✓ Number of features: {len(feature_cols)}")
        print(f"✓ Target range: [{y.min():.2f}, {y.max():.2f}]")
        print(f"✓ Target mean: {y.mean():.2f}, std: {y.std():.2f}")

        return X, y

    def train_and_evaluate_models(self, X, y, test_size=0.2, random_state=42):
        """Train and compare multiple regression models."""
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # Define models to compare
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10,
                                                   random_state=random_state, n_jobs=-1),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5,
                                                           learning_rate=0.1, random_state=random_state),
            'Ridge Regression': Ridge(alpha=1.0),
            'Lasso Regression': Lasso(alpha=0.1, max_iter=5000),
            'SVR (RBF)': SVR(kernel='rbf', C=10, gamma='scale')
        }

        results = {}

        print("\n" + "="*70)
        print("MODEL EVALUATION RESULTS")
        print("="*70)

        for name, model in models.items():
            print(f"\n{name}:")
            print("-" * 50)

            # Train model
            model.fit(X_train_scaled, y_train)

            # Predictions
            y_train_pred = model.predict(X_train_scaled)
            y_test_pred = model.predict(X_test_scaled)

            # Calculate metrics
            train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
            test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
            train_mae = mean_absolute_error(y_train, y_train_pred)
            test_mae = mean_absolute_error(y_test, y_test_pred)
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)

            # Cross-validation
            cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                       cv=5, scoring='neg_root_mean_squared_error')
            cv_rmse = -cv_scores.mean()

            results[name] = {
                'model': model,
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'train_r2': train_r2,
                'test_r2': test_r2,
                'cv_rmse': cv_rmse
            }

            print(f"  Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
            print(f"  Train MAE:  {train_mae:.4f}  | Test MAE:  {test_mae:.4f}")
            print(f"  Train R²:   {train_r2:.4f}  | Test R²:   {test_r2:.4f}")
            print(f"  CV RMSE:    {cv_rmse:.4f} (±{cv_scores.std():.4f})")

        # Find best model based on test RMSE
        best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse'])[0]
        self.model = results[best_model_name]['model']

        print("\n" + "="*70)
        print(f"BEST MODEL: {best_model_name}")
        print(f"Test RMSE: {results[best_model_name]['test_rmse']:.4f}")
        print(f"Test MAE:  {results[best_model_name]['test_mae']:.4f}")
        print(f"Test R²:   {results[best_model_name]['test_r2']:.4f}")
        print("="*70)

        return results, X_test_scaled, y_test

    def feature_importance_analysis(self, model_name, model, X):
        """Analyze feature importance for tree-based models."""
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1][:20]  # Top 20 features

            print(f"\nTop 20 Features for {model_name}:")
            print("-" * 50)
            for i, idx in enumerate(indices, 1):
                print(f"{i:2d}. {X.columns[idx]:30s} {importances[idx]:.4f}")

In [6]:
if __name__ == "__main__":
    r_path = "/content/R"
    g_path = "/content/G"
    b_path = "/content/B"

    predictor = RGBMentalPerformancePredictor(r_path, g_path, b_path)

    # Load and process data
    df = predictor.load_data()
    X, y = predictor.prepare_data()

    # Train and evaluate models
    results, X_test, y_test = predictor.train_and_evaluate_models(X, y)

    # Feature importance for best model
    best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse'])[0]
    best_model = results[best_model_name]['model']
    predictor.feature_importance_analysis(best_model_name, best_model, X)

    print("\n✓ Model training complete!")
    print(f"✓ Best model: {best_model_name}")
    print(f"✓ Ready for predictions on new RGB time series data")

Loading data from RGB channels...
Found 10 participants: ['10', '11', '12', '3', '4', '5', '6', '7', '8', '9']
Processing participant 10...
  Channel R: 1 chunk(s)
    Total rows: 135
  Channel G: 1 chunk(s)
    Total rows: 135
  Channel B: 1 chunk(s)
    Total rows: 135
Processing participant 11...
  Channel R: 1 chunk(s)
    Total rows: 87
  Channel G: 1 chunk(s)
    Total rows: 87
  Channel B: 1 chunk(s)
    Total rows: 87
Processing participant 12...
  Channel R: 1 chunk(s)
    Total rows: 85
  Channel G: 1 chunk(s)
    Total rows: 85
  Channel B: 1 chunk(s)
    Total rows: 85
Processing participant 3...
  Channel R: 1 chunk(s)
    Total rows: 92
  Channel G: 1 chunk(s)
    Total rows: 92
  Channel B: 1 chunk(s)
    Total rows: 92
Processing participant 4...
  Channel R: 1 chunk(s)
    Total rows: 74
  Channel G: 1 chunk(s)
    Total rows: 74
  Channel B: 1 chunk(s)
    Total rows: 74
Processing participant 5...
  Channel R: 1 chunk(s)
    Total rows: 120
  Channel G: 1 chunk(s)
  